# DSC@UIT 2026 — Full Legal RAG Pipeline (2×T4, M3)

Pipeline chính thức cho 1.000 câu public:

`BM25 Top-70 + E5 dense Top-70 → RRF Top-40 → reranker Top-5 → Vi-Qwen2-1.5B-RAG 4-bit, 640 tokens`.

Các giai đoạn GPU dùng hai process, mỗi process giữ một model trên một T4. Kết quả trung gian được lưu trong `/kaggle/working/legal_rag_full` để có thể tiếp tục trong cùng phiên Kaggle.


## 1. Setup

Chọn `Accelerator → GPU T4 ×2`, bật Internet hoặc attach sẵn các model. Dataset đã được Kaggle tự giải nén trong `/kaggle/input`.

In [ ]:
!pip install -q -U "transformers>=4.44" sentence-transformers accelerate bitsandbytes bm25s PyStemmer faiss-cpu
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Cấu hình M3

In [ ]:
from pathlib import Path
import glob, json, os, subprocess, time, gc
import numpy as np

SEED = 42
BM25_TOP_K = 70
DENSE_TOP_K = 70
RRF_TOP_K = 40
RERANK_TOP_K = 5
RRF_K = 60
MAX_INPUT_TOKENS = 6144
MAX_NEW_TOKENS = 640
GEN_BATCH_SIZE = 2

EMBED_MODEL = "intfloat/multilingual-e5-small"
RERANK_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
GEN_MODEL = "AITeamVN/Vi-Qwen2-1.5B-RAG"

WORK = Path("/kaggle/working/legal_rag_full")
WORK.mkdir(parents=True, exist_ok=True)

def find_one(name):
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert matches, f"Không tìm thấy {name} trong /kaggle/input"
    if len(matches) > 1: print(f"Cảnh báo: có {len(matches)} file {name}; dùng {matches[0]}")
    return Path(matches[0])

CHUNKS_PATH = find_one("context_chunks.jsonl")
PUBLIC_PATH = find_one("public_clean.jsonl")
assert CHUNKS_PATH.parent == PUBLIC_PATH.parent, "Context và public thuộc hai dataset khác nhau"
print("Chunks:", CHUNKS_PATH)
print("Public:", PUBLIC_PATH)

## 3. Đọc public và tạo catalog chunks

Catalog giữ metadata cần cho retrieval/reranking/generation. Đây là phần dùng nhiều RAM nhất; không nhân bản catalog sang các worker embedding.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

public_rows = load_jsonl(PUBLIC_PATH)
assert len(public_rows) == 1000

chunk_rows, retrieval_texts, chunk_ids = [], [], []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        chunk_rows.append(row)
        retrieval_texts.append(row["retrieval_text"])
        chunk_ids.append(row["chunk_id"])

assert len(chunk_rows) == len(set(chunk_ids))
print("Public:", len(public_rows), "Chunks:", len(chunk_rows))

## 4. Encode toàn bộ chunks trên hai T4

Mỗi worker đọc xen kẽ các dòng thuộc shard của mình. Embedding lưu `float16`; khi dựng FAISS sẽ đổi sang `float32`. Cell tự bỏ qua nếu checkpoint đã đủ.

In [ ]:
%%writefile /kaggle/working/legal_rag_full/embed_worker.py
import json, os, time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from sentence_transformers import SentenceTransformer

WORK = Path("/kaggle/working/legal_rag_full")
SOURCE = Path(os.environ["CHUNKS_PATH"])
MODEL_NAME = os.environ["EMBED_MODEL"]
dist.init_process_group("nccl")
rank, world = dist.get_rank(), dist.get_world_size()
local_rank = int(os.environ["LOCAL_RANK"]); torch.cuda.set_device(local_rank)

texts, indices = [], []
with open(SOURCE, encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if idx % world == rank:
            texts.append("passage: " + json.loads(line)["retrieval_text"])
            indices.append(idx)

model = SentenceTransformer(MODEL_NAME, device=f"cuda:{local_rank}")
start = time.perf_counter()
vectors = model.encode(texts, batch_size=256, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=(rank == 0))
np.save(WORK / f"context_emb_rank{rank}.npy", vectors.astype("float16"))
np.save(WORK / f"context_idx_rank{rank}.npy", np.asarray(indices, dtype="int64"))
json.dump({"rows":len(indices), "seconds":time.perf_counter()-start},
          open(WORK / f"embed_stats_rank{rank}.json", "w"))
dist.barrier(); dist.destroy_process_group()

In [ ]:
expected = len(chunk_rows)
ready = all((WORK / f"context_emb_rank{r}.npy").exists() for r in [0,1])
if ready:
    ready = sum(len(np.load(WORK / f"context_idx_rank{r}.npy", mmap_mode="r")) for r in [0,1]) == expected
if not ready:
    env = os.environ.copy()
    env.update({"CHUNKS_PATH":str(CHUNKS_PATH), "EMBED_MODEL":EMBED_MODEL})
    subprocess.run(["torchrun","--standalone","--nproc_per_node=2",str(WORK/"embed_worker.py")], check=True, env=env)
else:
    print("Dùng checkpoint embedding đã có")
print("Mục tiêu ngân sách: embedding + retrieval <= 90 phút")

## 5. Dense retrieval bằng FAISS

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

parts = []
for rank in [0,1]:
    idx = np.load(WORK / f"context_idx_rank{rank}.npy")
    emb = np.load(WORK / f"context_emb_rank{rank}.npy").astype("float32")
    parts.append((idx, emb))
dimension = parts[0][1].shape[1]
context_embeddings = np.empty((len(chunk_rows), dimension), dtype="float32")
for idx, emb in parts: context_embeddings[idx] = emb
del parts; gc.collect()

index = faiss.IndexFlatIP(dimension)
index.add(context_embeddings)
del context_embeddings; gc.collect()

query_encoder = SentenceTransformer(EMBED_MODEL, device="cuda:0")
query_vectors = query_encoder.encode(["query: "+x["question"] for x in public_rows],
    batch_size=128, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
dense_scores, dense_indices = index.search(query_vectors, DENSE_TOP_K)
del query_encoder, index, query_vectors
import torch; torch.cuda.empty_cache(); gc.collect()
print(dense_indices.shape)

## 6. BM25 retrieval và RRF

BM25 bắt tốt số hiệu, Điều/Khoản và cụm pháp lý; dense retrieval bắt tốt câu diễn đạt lại. Không cộng trực tiếp hai loại score vì khác thang đo.

In [ ]:
import bm25s

bm25_tokens = bm25s.tokenize(retrieval_texts, stopwords=None, show_progress=True)
bm25 = bm25s.BM25()
bm25.index(bm25_tokens, show_progress=True)
query_tokens = bm25s.tokenize([x["question"] for x in public_rows], stopwords=None)
bm25_indices, bm25_scores = bm25.retrieve(query_tokens, k=BM25_TOP_K, show_progress=True)

def rrf(dense_row, bm25_row, top_k=RRF_TOP_K, k=RRF_K):
    fused = {}
    for ranking in (dense_row, bm25_row):
        for rank, doc_idx in enumerate(ranking, 1):
            doc_idx = int(doc_idx)
            fused[doc_idx] = fused.get(doc_idx, 0.0) + 1.0/(k+rank)
    return sorted(fused, key=fused.get, reverse=True)[:top_k], fused

candidates = []
for qi, question in enumerate(public_rows):
    selected, fused = rrf(dense_indices[qi], bm25_indices[qi])
    candidates.append({
        "id": question["id"], "question": question["question"],
        "candidates": [{**chunk_rows[i], "rrf_score":fused[i]} for i in selected]
    })

with open(WORK/"retrieval_candidates.jsonl", "w", encoding="utf-8") as f:
    for row in candidates: f.write(json.dumps(row, ensure_ascii=False)+"\n")
assert len(candidates)==1000 and all(len(x["candidates"])==RRF_TOP_K for x in candidates)
print("Đã lưu", len(candidates)*RRF_TOP_K, "cặp rerank")

## 7. Rerank Top-40 → Top-5 trên hai T4

In [ ]:
%%writefile /kaggle/working/legal_rag_full/rerank_worker.py
import json, os, time
from pathlib import Path
import torch
import torch.distributed as dist
from sentence_transformers import CrossEncoder

WORK=Path("/kaggle/working/legal_rag_full")
dist.init_process_group("nccl")
rank, world=dist.get_rank(), dist.get_world_size()
local_rank=int(os.environ["LOCAL_RANK"]); torch.cuda.set_device(local_rank)
rows=[json.loads(x) for x in open(WORK/"retrieval_candidates.jsonl",encoding="utf-8")]
local=rows[rank::world]
model=CrossEncoder(os.environ["RERANK_MODEL"],device=f"cuda:{local_rank}",max_length=512)
outputs=[]; start=time.perf_counter()
for row in local:
    pairs=[[row["question"],c["retrieval_text"]] for c in row["candidates"]]
    scores=model.predict(pairs,batch_size=64,show_progress_bar=False)
    order=sorted(range(len(scores)),key=lambda i:float(scores[i]),reverse=True)[:int(os.environ["TOP_K"])]
    outputs.append({"id":row["id"],"question":row["question"],
        "contexts":[{**row["candidates"][i],"rerank_score":float(scores[i])} for i in order]})
with open(WORK/f"reranked_rank{rank}.jsonl","w",encoding="utf-8") as f:
    for x in outputs:f.write(json.dumps(x,ensure_ascii=False)+"\n")
json.dump({"rows":len(outputs),"seconds":time.perf_counter()-start},open(WORK/f"rerank_stats_rank{rank}.json","w"))
dist.barrier();dist.destroy_process_group()

In [ ]:
rerank_ready = all((WORK/f"reranked_rank{r}.jsonl").exists() for r in [0,1])
if not rerank_ready:
    env=os.environ.copy();env.update({"RERANK_MODEL":RERANK_MODEL,"TOP_K":str(RERANK_TOP_K)})
    subprocess.run(["torchrun","--standalone","--nproc_per_node=2",str(WORK/"rerank_worker.py")],check=True,env=env)
reranked=[]
for rank in [0,1]: reranked += load_jsonl(WORK/f"reranked_rank{rank}.jsonl")
position={x["id"]:i for i,x in enumerate(public_rows)}
reranked.sort(key=lambda x:position[x["id"]])
assert len(reranked)==1000 and len({x["id"] for x in reranked})==1000
with open(WORK/"reranked_all.jsonl","w",encoding="utf-8") as f:
    for x in reranked:f.write(json.dumps(x,ensure_ascii=False)+"\n")
print("Rerank hoàn tất")

## 8. Generation M3 trên hai T4

Mỗi GPU xử lý 500 câu. Worker ghi checkpoint mỗi 10 câu; chạy lại cell sẽ bỏ qua ID đã hoàn thành.

In [ ]:
%%writefile /kaggle/working/legal_rag_full/generate_worker.py
import json, os, re, time, unicodedata
from pathlib import Path
import torch
import torch.distributed as dist
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig

WORK=Path("/kaggle/working/legal_rag_full")
dist.init_process_group("nccl")
rank,world=dist.get_rank(),dist.get_world_size()
local_rank=int(os.environ["LOCAL_RANK"]);torch.cuda.set_device(local_rank)
rows=[json.loads(x) for x in open(WORK/"reranked_all.jsonl",encoding="utf-8")]
local=rows[rank::world];out_path=WORK/f"predictions_rank{rank}.jsonl"
done={}
if out_path.exists():
    for line in open(out_path,encoding="utf-8"):
        row=json.loads(line);done[row["id"]]=row
pending=[x for x in local if x["id"] not in done]

quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
tokenizer=AutoTokenizer.from_pretrained(os.environ["GEN_MODEL"])
tokenizer.padding_side="left"
if tokenizer.pad_token_id is None: tokenizer.pad_token_id=tokenizer.eos_token_id
model=AutoModelForCausalLM.from_pretrained(os.environ["GEN_MODEL"],quantization_config=quant,
    device_map={"":local_rank},torch_dtype=torch.float16).eval()

def make_prompt(row):
    blocks=[]
    for i,c in enumerate(row["contexts"][:5],1):
        blocks.append(f"VĂN BẢN {i}\nTên: {c['document_name']}\nMục: {c['section']}\nNội dung:\n{c['text']}")
    instruction=("Bạn là hệ thống trả lời câu hỏi pháp luật Việt Nam. Chỉ sử dụng các văn bản được cung cấp. "
      "Trả lời đầy đủ mọi nội dung liên quan trực tiếp; giữ nguyên thuật ngữ pháp lý, số hiệu, Điều, Khoản, Điểm, "
      "mức tiền, thời hạn và tên cơ quan. Nêu đủ điều kiện, ngoại lệ, hình thức xử phạt và biện pháp khắc phục nếu có. "
      "Sắp xếp theo thứ tự quy định; không dùng từ đồng nghĩa khi có thể giữ nguyên văn; không thêm kiến thức ngoài văn bản. "
      "Mở đầu bằng căn cứ pháp lý và kết thúc bằng kết luận trực tiếp bắt đầu bằng 'Như vậy,'.")
    return instruction+"\n\n"+"\n\n".join(blocks)+f"\n\nCÂU HỎI: {row['question']}\nTRẢ LỜI:"

def clean(text):
    text=unicodedata.normalize("NFC",text).strip()
    text=re.split(r"\n(?:CÂU HỎI|VĂN BẢN):",text,maxsplit=1)[0]
    text=re.sub(r"[ \t]+"," ",text);return re.sub(r"\n{3,}","\n\n",text).strip()

start=time.perf_counter();batch_size=int(os.environ.get("GEN_BATCH_SIZE","2"))
for offset in range(0,len(pending),batch_size):
    batch=pending[offset:offset+batch_size];prompts=[]
    for row in batch:
        prompt=make_prompt(row)
        if tokenizer.chat_template:
            prompt=tokenizer.apply_chat_template([{"role":"user","content":prompt}],tokenize=False,add_generation_prompt=True)
        prompts.append(prompt)
    inputs=tokenizer(prompts,return_tensors="pt",padding=True,truncation=True,
        max_length=int(os.environ["MAX_INPUT"]),pad_to_multiple_of=8).to(f"cuda:{local_rank}")
    with torch.inference_mode():
        output=model.generate(**inputs,max_new_tokens=int(os.environ["MAX_NEW"]),do_sample=False,num_beams=1,
            repetition_penalty=1.02,use_cache=True,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    prompt_width=inputs["input_ids"].shape[1]
    for i,row in enumerate(batch):
        answer=clean(tokenizer.decode(output[i,prompt_width:],skip_special_tokens=True))
        done[row["id"]]={"id":row["id"],"question":row["question"],"answer":answer,
            "context_ids":[c["chunk_id"] for c in row["contexts"][:5]]}
    completed=offset+len(batch)
    if completed%10==0 or completed==len(pending):
        with open(out_path,"w",encoding="utf-8") as f:
            for original in local:
                if original["id"] in done:f.write(json.dumps(done[original["id"]],ensure_ascii=False)+"\n")
        elapsed=time.perf_counter()-start
        eta=elapsed/max(completed,1)*(len(pending)-completed)
        print(f"rank={rank} {completed}/{len(pending)} elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m",flush=True)
json.dump({"total":len(local),"new":len(pending),"seconds":time.perf_counter()-start},open(WORK/f"generation_stats_rank{rank}.json","w"))
dist.barrier();dist.destroy_process_group()

In [ ]:
env=os.environ.copy();env.update({"GEN_MODEL":GEN_MODEL,"MAX_INPUT":str(MAX_INPUT_TOKENS),
    "MAX_NEW":str(MAX_NEW_TOKENS),"GEN_BATCH_SIZE":str(GEN_BATCH_SIZE)})
subprocess.run(["torchrun","--standalone","--nproc_per_node=2",str(WORK/"generate_worker.py")],check=True,env=env)

## 9. Gộp kết quả và tạo submission

In [ ]:
predictions=[]
for rank in [0,1]: predictions += load_jsonl(WORK/f"predictions_rank{rank}.jsonl")
position={x["id"]:i for i,x in enumerate(public_rows)}
predictions.sort(key=lambda x:position[x["id"]])
assert len(predictions)==1000 and len({x["id"] for x in predictions})==1000
assert all(x["answer"].strip() for x in predictions)

submission={x["id"]:{"question":x["question"],"answer":x["answer"]} for x in predictions}
with open(WORK/"submission.json","w",encoding="utf-8") as f:
    json.dump(submission,f,ensure_ascii=False,indent=2)
with open(WORK/"predictions_with_context.jsonl","w",encoding="utf-8") as f:
    for x in predictions:f.write(json.dumps(x,ensure_ascii=False)+"\n")

print("Submission:",WORK/"submission.json")
print("Số câu:",len(submission))
print("Độ dài đáp án trung bình (từ):",round(np.mean([len(x["answer"].split()) for x in predictions]),1))
print(json.dumps(predictions[0],ensure_ascii=False,indent=2)[:2500])

## 10. Tệp cần tải xuống

- `submission.json`: file nộp.
- `predictions_with_context.jsonl`: dùng kiểm tra căn cứ và phân tích lỗi.
- Giữ `retrieval_candidates.jsonl`, `reranked_all.jsonl` và hai shard embedding nếu muốn thử generator khác mà không chạy lại retrieval.